# 한국어 챗봇 모델 학습 (polyglot-ko-1.3b + LoRA)

13억 파라미터 한국어 모델을 LoRA 로 파인튜닝한다. KoGPT2 보다 크고 품질이 높다.

런타임 → 런타임 유형 변경 → GPU(T4) 설정 후 실행.

In [ ]:
!pip -q install transformers datasets accelerate peft
!pip -q uninstall -y torchao

In [ ]:
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from datasets import load_dataset
dataset = load_dataset('beomi/KoAlpaca-v1.1a', split='train')
dataset = dataset.select(range(12000))
print(dataset, dataset[0])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
BASE = 'EleutherAI/polyglot-ko-1.3b'
tokenizer = AutoTokenizer.from_pretrained(BASE)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16)

rt = tokenizer.decode(tokenizer.encode('파이썬이 뭐야?'), skip_special_tokens=True)
print('토크나이저 한국어 테스트:', repr(rt))
assert '파이썬' in rt.replace(' ',''), '토크나이저 이상'
print('파라미터 수:', model.num_parameters())

In [ ]:
PROMPT = '### 질문: {q}\n### 답변: {a}'
MAX_LEN = 256
def fmt(ex):
    text = PROMPT.format(q=ex['instruction'].strip(), a=str(ex['output']).strip()) + tokenizer.eos_token
    return tokenizer(text, truncation=True, max_length=MAX_LEN)
tokenized = dataset.map(fmt, remove_columns=dataset.column_names)
print(tokenized)

In [ ]:
from peft import LoraConfig, get_peft_model
lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                  target_modules=['query_key_value'], task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
args = TrainingArguments(
    output_dir='out', per_device_train_batch_size=4, gradient_accumulation_steps=4,
    num_train_epochs=3, learning_rate=2e-4, fp16=True,
    warmup_ratio=0.05, logging_steps=50, save_strategy='no', report_to='none')
Trainer(model=model, args=args, train_dataset=tokenized, data_collator=collator).train()

In [ ]:
# LoRA 어댑터를 본체에 병합해 일반 모델로 저장 (추론 시 peft 불필요)
merged = model.merge_and_unload()
SAVE = 'polyglot-ko-chatbot'
merged.save_pretrained(SAVE); tokenizer.save_pretrained(SAVE)

def ask(q, n=80):
    p = '### 질문: ' + q + '\n### 답변:'
    ids = tokenizer.encode(p, return_tensors='pt').to(merged.device)
    out = merged.generate(ids, max_new_tokens=n, do_sample=True, top_p=0.92,
                          temperature=0.8, repetition_penalty=1.2,
                          pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).split('###')[0].strip()
for q in ['파이썬이 뭐야?', '건강하게 사는 법 알려줘', '딥러닝과 머신러닝 차이는?']:
    print('Q:', q); print('A:', ask(q)); print()

In [ ]:
from huggingface_hub import login, create_repo
login(token='hf_본인_WRITE_토큰')
REPO = 'jjminu/polyglot-ko-chatbot'
create_repo(REPO, exist_ok=True)
merged.push_to_hub(REPO); tokenizer.push_to_hub(REPO)
print('업로드 완료:', REPO)